In [1]:
# Install openpyxl to handle .xlsm files and pandas for data manipulation
!pip install pandas openpyxl

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import sys
# This ensures we install the library in the same environment as this notebook
!{sys.executable} -m pip install pandas openpyxl

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [3]:
import sys
# This command forces the installation into the specific Python path Jupyter is using
!{sys.executable} -m pip install openpyxl pandas

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [1]:
import pandas as pd
import os

# Define file path
file_path = '../data/raw/FINAL COMPARABLE SPREADSHEET 2026.xlsm'

# 1. Check if the file is reachable
if not os.path.exists(file_path):
    print(f"OUTCOME: File not found at {file_path}")
else:
    print("OUTCOME: File found successfully.")
    
    # 2. Try to load the sheet names
    try:
        # We specify the engine 'openpyxl' to handle the .xlsm format
        xl = pd.ExcelFile(file_path, engine='openpyxl')
        print(f"OUTCOME: Success! Sheets found: {xl.sheet_names}")
        
        # 3. Peek at the first sheet
        df = pd.read_excel(file_path, sheet_name=0, nrows=5, engine='openpyxl')
        print("\n--- Data Column Names ---")
        print(df.columns.tolist())
        
    except Exception as e:
        print(f"OUTCOME: An error occurred: {e}")

OUTCOME: File found successfully.
OUTCOME: Success! Sheets found: ['Sheet1', 'Suggestion1', 'Suggestion2', 'Suggestion3']

--- Data Column Names ---
['Property Name', 'Address', 'Parish', 'Price Sold', 'Date', 'Sq. ft.', 'Bed ', 'Bath', 'Lot Size', 'No. Units', 'ARV', 'Assessment', 'Type', 'Guest', 'Pool', 'Waterfront', 'Listed', 'Zone', 'Notes']


In [1]:
import pandas as pd
import sqlite3
import os

# File paths
input_file = '../data/raw/FINAL COMPARABLE SPREADSHEET 2026.xlsm'
db_path = '../data/processed/propiedades_comparables.db'

# 1. Load the data
# We use engine='openpyxl' for .xlsm files
df = pd.read_excel(input_file, sheet_name=0, engine='openpyxl')

# 2. Basic Cleaning (Transform)
# Rename columns to match our SQL table (Snake Case)
mapping = {
    'DATE': 'sale_date',
    'ZONING': 'zoning',
    'TYPE': 'property_type',
    'LOT SIZE': 'lot_size_sqft',
    'BLDG SIZE': 'bldg_size_sqft',
    'SALE PRICE': 'sale_price',
    'PSF (BLDG)': 'price_per_sqft',
    'REMARKS': 'remarks'
}
df = df.rename(columns=mapping)

# 3. Splitting 'NAME / PARISH' into two columns
# We split by the '/' character. Expand=True creates two columns.
df[['property_name', 'parish_region']] = df['NAME / PARISH'].str.split('/', n=1, expand=True)

# 4. Clean Numeric Data
# This removes currency symbols and commas so SQL can treat them as numbers
cols_to_fix = ['sale_price', 'price_per_sqft', 'lot_size_sqft', 'bldg_size_sqft']
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Load into SQLite (DBeaver)
conn = sqlite3.connect(db_path)
# We use if_exists='append' because the table already exists
df_final = df[['sale_date', 'property_name', 'parish_region', 'zoning', 
               'property_type', 'lot_size_sqft', 'bldg_size_sqft', 
               'sale_price', 'price_per_sqft', 'remarks']]

df_final.to_sql('comparables', conn, if_exists='append', index=False)
conn.close()

print("ETL Process Complete! Check DBeaver to see the data.")

KeyError: 'NAME / PARISH'

In [2]:
import pandas as pd
import sqlite3
import os

# File paths
input_file = '../data/raw/FINAL COMPARABLE SPREADSHEET 2026.xlsm'
db_path = '../data/processed/propiedades_comparables.db'

# 1. Load the data
df = pd.read_excel(input_file, sheet_name=0, engine='openpyxl')

# 2. Splitting 'NAME / PARISH' FIRST
# We do this while the original column name still exists
if 'NAME / PARISH' in df.columns:
    df[['property_name', 'parish_region']] = df['NAME / PARISH'].str.split('/', n=1, expand=True)
else:
    print("Warning: 'NAME / PARISH' column not found. Check for extra spaces.")

# 3. Rename other columns (Transform)
mapping = {
    'DATE': 'sale_date',
    'ZONING': 'zoning',
    'TYPE': 'property_type',
    'LOT SIZE': 'lot_size_sqft',
    'BLDG SIZE': 'bldg_size_sqft',
    'SALE PRICE': 'sale_price',
    'PSF (BLDG)': 'price_per_sqft',
    'REMARKS': 'remarks'
}
df = df.rename(columns=mapping)

# 4. Clean Numeric Data
# We use errors='coerce' to turn messy text into NaN (null) numbers safely
cols_to_fix = ['sale_price', 'price_per_sqft', 'lot_size_sqft', 'bldg_size_sqft']
for col in cols_to_fix:
    # Remove symbols like '$' or ',' before converting
    df[col] = df[col].replace(r'[\$,]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Load into SQLite
conn = sqlite3.connect(db_path)

# Select only the columns that exist in your DBeaver table
final_columns = [
    'sale_date', 'property_name', 'parish_region', 'zoning', 
    'property_type', 'lot_size_sqft', 'bldg_size_sqft', 
    'sale_price', 'price_per_sqft', 'remarks'
]

# Check which columns actually made it into the dataframe
existing_columns = [c for c in final_columns if c in df.columns]

df[existing_columns].to_sql('comparables', conn, if_exists='append', index=False)
conn.close()

print("ETL Process Complete! The data is now in your SQLite database.")

KeyError: 'sale_price'

In [3]:
Warning: 'NAME / PARISH' column not found. Check for extra spaces.

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File /usr/lib/python3/dist-packages/pandas/core/indexes/base.py:3791, in Index.get_loc(self, key)
   3790 try:
-> 3791     return self._engine.get_loc(casted_key)
   3792 except KeyError as err:

File /usr/lib/python3/dist-packages/pandas/_libs/index.pyx:152, in pandas._libs.index.IndexEngine.get_loc()

File /usr/lib/python3/dist-packages/pandas/_libs/index.pyx:181, in pandas._libs.index.IndexEngine.get_loc()

File pandas/_libs/hashtable_class_helper.pxi:7080, in pandas._libs.hashtable.PyObjectHashTable.get_item()

File pandas/_libs/hashtable_class_helper.pxi:7088, in pandas._libs.hashtable.PyObjectHashTable.get_item()

KeyError: 'sale_price'

The above exception was the direct cause of the following exception:

KeyError                                  Traceback (most recent call last)
Cell In[2], line 37
     34 cols_to_fix = ['sale_price', 'price_per_sqft', 'lot_size_sqft', 'bldg_size_sqft']
     35 for col in cols_to_fix:
     36     # Remove symbols like '$' or ',' before converting
---> 37     df[col] = df[col].replace(r'[\$,]', '', regex=True)
     38     df[col] = pd.to_numeric(df[col], errors='coerce')
     40 # 5. Load into SQLite

File /usr/lib/python3/dist-packages/pandas/core/frame.py:3893, in DataFrame.__getitem__(self, key)
   3891 if self.columns.nlevels > 1:
   3892     return self._getitem_multilevel(key)
-> 3893 indexer = self.columns.get_loc(key)
   3894 if is_integer(indexer):
   3895     indexer = [indexer]

File /usr/lib/python3/dist-packages/pandas/core/indexes/base.py:3798, in Index.get_loc(self, key)
   3793     if isinstance(casted_key, slice) or (
   3794         isinstance(casted_key, abc.Iterable)
   3795         and any(isinstance(x, slice) for x in casted_key)
   3796     ):
   3797         raise InvalidIndexError(key)
-> 3798     raise KeyError(key) from err
   3799 except TypeError:
   3800     # If we have a listlike key, _check_indexing_error will raise
   3801     #  InvalidIndexError. Otherwise we fall through and re-raise
   3802     #  the TypeError.
   3803     self._check_indexing_error(key)

KeyError: 'sale_price'



SyntaxError: invalid syntax (3218896466.py, line 1)

In [4]:
import pandas as pd
import sqlite3
import os

# File paths
input_file = '../data/raw/FINAL COMPARABLE SPREADSHEET 2026.xlsm'
db_path = '../data/processed/propiedades_comparables.db'

# 1. Load the data
df = pd.read_excel(input_file, sheet_name=0, engine='openpyxl')

# 2. Safety Check: Clean column names (Remove leading/trailing spaces)
# Sometimes 'NAME / PARISH' is actually ' NAME / PARISH '
df.columns = df.columns.str.strip()

# 3. Splitting 'NAME / PARISH'
if 'NAME / PARISH' in df.columns:
    # We split and create two new columns
    df[['property_name', 'parish_region']] = df['NAME / PARISH'].str.split('/', n=1, expand=True)
    print("SUCCESS: 'NAME / PARISH' column split correctly.")
else:
    print("ERROR: 'NAME / PARISH' not found.")
    print(f"Current columns are: {df.columns.tolist()}")

# 4. Rename other columns
mapping = {
    'DATE': 'sale_date',
    'ZONING': 'zoning',
    'TYPE': 'property_type',
    'LOT SIZE': 'lot_size_sqft',
    'BLDG SIZE': 'bldg_size_sqft',
    'SALE PRICE': 'sale_price',
    'PSF (BLDG)': 'price_per_sqft',
    'REMARKS': 'remarks'
}
df = df.rename(columns=mapping)

# 5. Clean Numeric Data (Remove $ and ,)
cols_to_fix = ['sale_price', 'price_per_sqft', 'lot_size_sqft', 'bldg_size_sqft']
for col in cols_to_fix:
    if col in df.columns:
        df[col] = df[col].replace(r'[\$,]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 6. Load into SQLite
conn = sqlite3.connect(db_path)
final_columns = [
    'sale_date', 'property_name', 'parish_region', 'zoning', 
    'property_type', 'lot_size_sqft', 'bldg_size_sqft', 
    'sale_price', 'price_per_sqft', 'remarks'
]

# Filter to keep only the columns that exist in our DataFrame
existing_columns = [c for c in final_columns if c in df.columns]

df[existing_columns].to_sql('comparables', conn, if_exists='append', index=False)
conn.close()

print("\n--- DONE ---")
print(f"Processed {len(df)} rows into the database.")

ERROR: 'NAME / PARISH' not found.
Current columns are: ['Property Name', 'Address', 'Parish', 'Price Sold', 'Date', 'Sq. ft.', 'Bed', 'Bath', 'Lot Size', 'No. Units', 'ARV', 'Assessment', 'Type', 'Guest', 'Pool', 'Waterfront', 'Listed', 'Zone', 'Notes']


OperationalError: near ")": syntax error

In [5]:
import pandas as pd
import sqlite3
import os

# File paths
input_file = '../data/raw/FINAL COMPARABLE SPREADSHEET 2026.xlsm'
db_path = '../data/processed/propiedades_comparables.db'

# 1. Load the data
df = pd.read_excel(input_file, sheet_name=0, engine='openpyxl')
df.columns = df.columns.str.strip() # Remove any hidden spaces

# 2. Define the correct mapping based on your actual Excel columns
mapping = {
    'Date': 'sale_date',
    'Property Name': 'property_name',
    'Parish': 'parish_region',
    'Zone': 'zoning',
    'Type': 'property_type',
    'Lot Size': 'lot_size_sqft',
    'Sq. ft.': 'bldg_size_sqft',
    'Price Sold': 'sale_price',
    'Notes': 'remarks'
}

# 3. Rename columns
df = df.rename(columns=mapping)

# 4. Clean Numeric Data
# We clean these so we can do math with them
cols_to_fix = ['sale_price', 'lot_size_sqft', 'bldg_size_sqft']
for col in cols_to_fix:
    if col in df.columns:
        # Remove $, commas, and whitespace
        df[col] = df[col].astype(str).replace(r'[\$,\s]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Calculate Price per Sq Ft (Value Add)
# If we have price and size, let's calculate the ratio
df['price_per_sqft'] = df['sale_price'] / df['bldg_size_sqft']

# 6. Final Selection (Matching your DBeaver table exactly)
final_columns = [
    'sale_date', 'property_name', 'parish_region', 'zoning', 
    'property_type', 'lot_size_sqft', 'bldg_size_sqft', 
    'sale_price', 'price_per_sqft', 'remarks'
]

# Ensure we only take columns that exist in our cleaned DataFrame
df_to_save = df[[c for c in final_columns if c in df.columns]]

# 7. Load into SQLite
conn = sqlite3.connect(db_path)
df_to_save.to_sql('comparables', conn, if_exists='append', index=False)
conn.close()

print(f"SUCCESS: {len(df_to_save)} rows inserted into the database.")

SUCCESS: 1540 rows inserted into the database.
